## 导入依赖

In [1]:
import math
import numpy as np
import yaml
# 自定义库 
from transform import Transform
from icp import icp_solver_svd
from arm5dof_uservo import Arm5DoFUServo
from config import *

## 载入九点标定的配置文件

In [2]:
# 载入九点标定法的配置文件
handeye_9points_config = None
with open("config/handeye_9points.yaml", 'r', encoding='utf-8') as f:
    handeye_9points_config = yaml.load(f.read(), Loader=yaml.SafeLoader)

## 创建机械臂

In [3]:
# 创建机械臂
arm = Arm5DoFUServo(is_init_pose=False)

In [4]:
# 设置为阻尼模式
arm.set_damping(0)

## 获取采样点

将机械臂末端拖拽到9个点的位置，记录下机械臂末端的坐标值(x, y, z)

![](./image/p0.jpg)

In [5]:
p0 = arm.get_tool_pose()[:3]
p0

[296.2516932300118, 47.55235902840156, 11.923904786301392]

![](./image/p1.jpg)

In [8]:
p1 = arm.get_tool_pose()[:3]
p1

[292.138925569875, -35.55103511045452, 11.239935570270632]

![](./image/p2.jpg)

In [9]:
p2 = arm.get_tool_pose()[:3]
p2

[290.7957422571215, -109.9747033562267, 14.847549065941166]

![](./image/p3.jpg)

In [10]:
p3 = arm.get_tool_pose()[:3]
p3

[231.38425059381686, 49.57291900791795, -4.051556368693895]

In [11]:
p4 = arm.get_tool_pose()[:3]
p4

[228.45288226886169, -33.433561501287215, -7.650197763309933]

![](./image/p5.jpg)

In [12]:
p5 = arm.get_tool_pose()[:3]
p5

[231.20703891613857, -111.62530715929641, 2.7240027451064464]

![](./image/p6.jpg)

In [13]:
p6 = arm.get_tool_pose()[:3]
p6

[171.51799827578114, 44.8997230726114, -5.128026794088811]

![](./image/p7.jpg)

In [14]:
p7 = arm.get_tool_pose()[:3]
p7

[168.97708675246423, -27.123082434461928, -1.9408878026326022]

![](./image/p8.jpg)

In [15]:
p8 = arm.get_tool_pose()[:3]
p8

[166.50034824417656, -107.74495627460067, -5.203780365622521]

## 数据保存

In [16]:
ws_9p_at_arm = np.float32([p0, p1, p2, p3, p4, p5, p6, p7, p8])
ws_9p_at_arm

array([[ 296.252,   47.552,   11.924],
       [ 292.139,  -35.551,   11.24 ],
       [ 290.796, -109.975,   14.848],
       [ 231.384,   49.573,   -4.052],
       [ 228.453,  -33.434,   -7.65 ],
       [ 231.207, -111.625,    2.724],
       [ 171.518,   44.9  ,   -5.128],
       [ 168.977,  -27.123,   -1.941],
       [ 166.5  , -107.745,   -5.204]], dtype=float32)

将数据存放到csv文件中

In [17]:
np.savetxt('config/arm9points.txt', ws_9p_at_arm, delimiter=',', fmt='%.1f')

## Base标定

In [18]:
# 设置Numpy的打印选项
# 精确位数3，不启用科学计数法
np.set_printoptions(precision=3, suppress=True)

# 载入九点标定法的配置文件
handeye_9points_config = None
with open("config/handeye_9points.yaml", 'r', encoding='utf-8') as f:
	handeye_9points_config = yaml.load(f.read(), Loader=yaml.SafeLoader)


# 载入机械臂末端达到九个点
# 载入九点在2D图像中的坐标
arm_9points = np.loadtxt("config/arm9points.txt", delimiter=',')
# 主要注意的是，对于关节臂构型的机械臂
# 有的在Z轴上的误差很大，因此需要将
# arm_9points里面Z轴坐标替换为Z轴的均值。
z_mean = np.mean(arm_9points[:, 2])

# z轴误差纠正拟合
# - 转化为r跟z的形式
r = np.sqrt(arm_9points[:, 0]**2 + arm_9points[:, 1]**2)
z_error =  arm_9points[:, 2] - z_mean
z_error_polyfit = np.polyfit(r, z_error, 1)
np.savetxt("config/z_error_polyfit.txt", z_error_polyfit, fmt='%.4f', delimiter=",")
arm_9points[:, 2] = z_mean
print(f"9点在机械臂坐标系的坐标: \n{arm_9points}")

# 9点在工作台坐标系的坐标
# ws_9points_df = pd.read_excel("config/九点坐标-工作台坐标系.xlsx")
# ws_9points = np.float64(ws_9points_df.to_numpy())[:, 1:]
w = handeye_9points_config["board_width"]
h = handeye_9points_config["board_height"]
# P0点的坐标
x0 = 0.5 * h
y0 = 0.5 * w
ws_9points = np.float64([
	[x0, y0, 0], 	#P0
	[x0, 0, 0], 	#P1
	[x0, -y0, 0], 	#P2
 	[0, y0, 0], 	#P3
	[0, 0, 0], 		#P4
	[0, -y0, 0], 	#P5
	[-x0, y0, 0], 	#P6
	[-x0, 0, 0], 	#P7
	[-x0, -y0, 0], 	#P8
])
print(f"9点在工作台坐标系的坐标: \n{ws_9points}")

# 求解机械臂基坐标系到工作台坐标系的变换
R, t = icp_solver_svd(arm_9points, ws_9points)
T_arm2ws = np.eye(4)
T_arm2ws[:3, :3] = R
T_arm2ws[:3, 3] = t.reshape(-1)
print("T_arm2ws: \n{}".format(T_arm2ws))
np.savetxt("config/T_arm2ws.txt", T_arm2ws, fmt='%.3f', delimiter=",")

9点在机械臂坐标系的坐标: 
[[ 296.3     47.6      1.844]
 [ 292.1    -35.6      1.844]
 [ 290.8   -110.       1.844]
 [ 231.4     49.6      1.844]
 [ 228.5    -33.4      1.844]
 [ 231.2   -111.6      1.844]
 [ 171.5     44.9      1.844]
 [ 169.     -27.1      1.844]
 [ 166.5   -107.7      1.844]]
9点在工作台坐标系的坐标: 
[[ 59.5  79.5   0. ]
 [ 59.5   0.    0. ]
 [ 59.5 -79.5   0. ]
 [  0.   79.5   0. ]
 [  0.    0.    0. ]
 [  0.  -79.5   0. ]
 [-59.5  79.5   0. ]
 [-59.5   0.    0. ]
 [-59.5 -79.5   0. ]]
T_arm2ws: 
[[  1.      0.022   0.    230.811]
 [ -0.022   1.      0.    -31.478]
 [  0.      0.      1.      1.844]
 [  0.      0.      0.      1.   ]]
